In [2]:
import json
import pandas as pd

def validate_and_clean_data(input_file, output_file):
    print(f"Loading data from {input_file}\n")
    
    # 1. Load the JSONL file
    data = []
    with open(input_file, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
            
    df = pd.DataFrame(data)
    initial_count = len(df)
    
    # 2. Check for missing critical data
    missing_titles = df['title'].isna().sum()
    missing_urls = df['url'].isna().sum()
    print(f"Quality Check: Found {missing_titles} missing titles and {missing_urls} missing URLs.")
    
    # 3. Drop exact duplicates (same URL or same exact Title)
    df_cleaned = df.drop_duplicates(subset=['url'], keep='first')
    df_cleaned = df_cleaned.drop_duplicates(subset=['title'], keep='first')
    
    # 4. Filter out garbage summaries (e.g., less than 10 characters)
    # Some APIs return "No summary" or empty strings. We want actual text.
    df_cleaned['summary_length'] = df_cleaned['summary'].astype(str).apply(len)
    df_cleaned = df_cleaned[df_cleaned['summary_length'] > 10]
    
    final_count = len(df_cleaned)
    dropped_count = initial_count - final_count
    
    print(f"\n Cleaning Results")
    print(f"Initial Documents: {initial_count}")
    print(f"Dropped Documents: {dropped_count} (Duplicates or empty summaries)")
    print(f"Final Clean Documents: {final_count}")
    
    # 5. Save the cleaned data back to a new JSONL
    # Drop the temporary length column first
    df_cleaned = df_cleaned.drop(columns=['summary_length'])
    df_cleaned.to_json(output_file, orient='records', lines=True)
    print(f"\nCleaned data saved to {output_file}")
    
    return final_count

if __name__ == "__main__":
    validate_and_clean_data("nvidia_raw_intelligence.jsonl", "nvidia_clean_intelligence.jsonl")

Loading data from nvidia_raw_intelligence.jsonl

Quality Check: Found 0 missing titles and 0 missing URLs.

 Cleaning Results
Initial Documents: 160
Dropped Documents: 40 (Duplicates or empty summaries)
Final Clean Documents: 120

Cleaned data saved to nvidia_clean_intelligence.jsonl
